# Processing pipeline

- [ ] Stage 1. Correct background
    - inputs:
        - `fov/`: folder containing all the field of views extracted from the experiments.
        - `background/`: folder containing a background image that will be used to correct the background in all the field of views.
        - `proc_metadata/`: folder containing csv files to track the process.

    - outputs:

        - `fov_corrected/`: folder containing the images after background corrections.
        - `background_correction_metadata.csv`: a file containing metadata about the background correction process.

## Imports

In [ ]:
# Built-in imports
import logging


# Third-party imports
from pathlib import Path
from omegaconf import OmegaConf

# Package imports
from acid.image_processing.background.load_background_function import (
    load_background_function,
)
from acid.utils.filesystem.filesystem import create_output_directories
from acid.utils.metadata.loading import load_metadata
from acid.utils.metadata.filtering import filter_metadata_by_splits


# ----- Temporal (delete once refactored)
import numpy as np
from pprint import pprint

## Configure logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

## Load Config File

In [ ]:
config = OmegaConf.load("config.yaml")

In [ ]:
# print(OmegaConf.to_yaml(config))

# Workflow

## 1. Create output and secondary output directory

In [ ]:
output_path, secondary_output_path = create_output_directories(
    output_directory=config["output"]["directory"],
    secondary_output_directory=config["output"]["secondary"]["directory"],
    enable_secondary_output=config["output"]["secondary"]["enabled"],
)

## 2. Open the metadata dataframe

In [ ]:
metadata_df, metadata_file_name = load_metadata(config["metadata"])

## 3. Select train data

In [ ]:
metadata_df = filter_metadata_by_splits(
    metadata_df, selected_splits="train", split_config=config.metadata.dataset_split
)

In [ ]:
metadata_df.info()

## 4. Read background data

### 4.3 New code

In [ ]:
backgrounds, background_file_names = load_background_function(
    background_config=config.background_correction,
    metadata_df=metadata_df,
)

In [ ]:
backgrounds.shape

In [ ]:
background_file_names

## 5. Main loop


### 5.1 Config

In [ ]:
# =============================================================================
# INPUT FIELD-OF-VIEW CONFIGURATION
# =============================================================================

# field of view column name in the metadata dataframe
fov_column_name = "ome_tif_file_name"

# indicate the path to the directory storing the fields of view
fov_directory = "data/proc/fov"

# within well position column name in the metadata dataframe
# this is used to identify the position of the field of view within the well
# 49 fields of view were acquired per each well, in a 7x7 grid
gridpos_column_name = "scene_name"

# well column name in the metadata dataframe
well_column_name = "well"


# =============================================================================
# GENERAL PROCESSING METADATA
# =============================================================================

# the data type of the image corrected for illumination
output_dtype = np.float32

# what is expected to be and what will be used as null value
null_value = np.nan

# --- parameters for saving metadata within the saved fields of view after illumination correction ---
# illumination correction image - name of processing date in metadata - this is the name
# of the entry in the metadata dictionary to save within the processed image.
# The entry indicates the date when the background correction was done.
proc_img_meta_date_name = "processing_date_yymmdd"

# illumination correction image - data type in metadata - this is the entry to use for indicating
# the data type of the illumination corrected image in the metadata saved within the processed image
proc_img_meta_dtype_name = "dtype"

# illumination correction image - date format in metadata - this is the format to use for indicating
# the date when the background correction was done in the metadata saved within the processed image
processing_date_format = "%y%m%d"


# =============================================================================
# ILLUMINATION CORRECTION IMAGE METADATA ENTRIES
# =============================================================================

# illumination correction image - illumination correction method in metadata - this is the entry to use for indicating
# the method used for illumination correction in the metadata saved within the processed image
illum_corr_method_metadata_entry = "illumination_correction_method"

# illumination correction image - illumination correction offset in metadata - this is the entry to use for indicating
# the offset used for illumination correction in the metadata saved within the processed image
illum_corr_offset_metadata_entry = "illumination_correction_offset"

# illumination correction image - illumination correction rescaling in metadata - this is the entry to use for indicating
# in the metadata saved within the processed image, whether or not the background function is rescaled when doing illumination correction
illum_corr_rescale_metadata_entry = "illumination_correction_rescale_background"

# illumination correction image - illumination correction clipping in metadata - this is the entry to use for indicating
# in the metadata saved within the processed image, whether or not the illumination corrected image is clipped
illum_corr_clipping_metadata_entry = "illumination_correction_clipping"

# illumination correction image - illumination correction clipping min value in metadata - this is the entry to use for
# indicating in the metadata saved within the processed image, what was the min value used for clipping (if clipping was performed)
illum_corr_clip_min_value_metadata_entry = "illumination_correction_min_clip_value"

# illumination correction image - illumination correction clipping max value in metadata - this is the entry to use for
# indicating in the metadata saved within the processed image, what was the max value used for clipping (if clipping was performed)
illum_corr_clip_max_value_metadata_entry = "illumination_correction_max_clip_value"

# illumination correction image - illumination correction offset_background in metadata - this is the entry to use for
# indicating in the metadata saved within the processed image, if the offset is subtracted to the background function
illum_corr_offset_background_metadata_entry = (
    "illumination_correction_offset_background"
)


# =============================================================================
# ILLUMINATION CORRECTION METHOD PARAMETERS
# =============================================================================
# background function computation strategy - this is the strategy used in part4a notebook for
# calculating the background function. Possible options are: 1,2,3.
# 1 - calculate a  background function per channel using all the fields of view in the dataset which are not flagged
# 2 - calculate a background function per each channel and well using the fields of view in the train set which are not
# flagged and belong to the well
# 3 - calculate a background function per each channel and grid position using the fields of view in the train set which are
# not flagged and belong to the same position (aka grid position) across different wells
background_function_strategy = 1

# the method used for illumination correction - this can be either "subtraction" or "division"
# method="subtraction"
method = "division"

# the offset to subtract from the image before correcting for the illumination
offset = 400

# whether or not to rescale the background function before carrying out illumination correction.
# Three options are available:
# 1) None -> no rescaling is done.
# 2) "max" -> the background function is divided by the max value. The resulting, normalized background function has 1 as max value.
# 3) "minmax" -> the background function is min-max normalized (background - min) / (max - min + epsilon)
rescale_background = "max"

# whether or not to clip the image corrected for illumination. This is a boolean variable, it can be True or False.
# If True, the image after illumination correction is clipped in the min_clip_value, max_clip_value range.
# For example: when subtraction is use the result can contain negative values. It is possible to set them to 0
# by setting clip_corrected_image=True and min_clip_value=0 (irrespective of the max_clip_value)
clip_corrected_image = False

# the min value to which illumination corrected image is clipped to if clip_corrected_image is set to True
min_clip_value = 0

# the max value to which illumination corrected image is clipped to if clip_corrected_image is set to True
max_clip_value = None

# whether or not to offset the background function before carrying out illumination correction.
# Two options are available: True or False
offset_background = True


# =============================================================================
# BACKGROUND CORRECTION ARRAY / NUMERICAL PARAMETERS
# =============================================================================

# --- parameters for background illumination correction ---
# channel axis to be passed to background calculating functions in order to calculate background functions per channel
# this is the axis along which the channels are organized in the field of view arrays.
# For example, if the field of view arrays have shape (channels, height, width), the channel axis is 0.
channel_axis = 0

# the small constant to add to the background in oder to avoid zero numbers in the background function, which
# would lead to mathematical instability (e.g. division by zero)
epsilon = 1e-8

# the data type to use for the mathematical operations (subtraction and division) during illumination correction.
# NOTE: the same data type is used for both the image to correct and the background function
working_dtype = np.float32

# additional parameters to pass to the initial container of the illumation corrected image during the correction process
# ref to the function correct_background within the image_processing.correct_background.py script for further details.
zero_kwargs = None

# select if printing process information during background correction
verbose = True


# =============================================================================
# OUTPUT IMAGE FILE NAMING AND SAVING
# =============================================================================

# ome suffix - used to save ome.tif files
ome_suffix = ".ome.tif"

# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = "_"

# illumination correction image name - savingword - this is the word to use in the field of view file name
# to indicate that the file is an illumination corrected field of view
fov_illumin_corrected_savingword = "bg"

# # indicate the path to the directory where outputs will be saved
# NOTE: it the directory does not exist, the pipeline will try to create it
output_directory = "data/proc/fov_proc"

# indicate whether to save the background function image with ImageJ compatible
save_imagej_compatible = True

# indicate the photometric interpretation to be used when saving the ome.tif files
# NOTE: at the moment, only 'minisblack' has been tested
photometric = "minisblack"


# =============================================================================
# METADATA DATAFRAME COLUMN NAMING
# =============================================================================

# --- parameters for metadata dataframe updating ---
# column name word separator - this is the separator to use for separating the different parts of
# the column names to be added to the metadata dataframe
column_name_separator = "_"

# illumination correction metadata dataframe - computation date column name - this is the name of the column
# to be added to the metadata dataframe to indicate the day when the illumination correction was computed.
illum_correct_df_date_clm_name = (
    f"illumination{column_name_separator}correction{column_name_separator}date"
)

# illumination correction metadata dataframe - corrected file column name - this is the name of the column
# to be added to the metadata dataframe to indicate the name used for saving the illumination corrected file.
illum_correct_df_file_name_clm_name = f"illumination{column_name_separator}correction{column_name_separator}file{column_name_separator}name"

# illumination correction metadata dataframe - method for illumination correction column name - this is the name
# of the column to be added to the metadata dataframe to indicate the method used for illumination correction.
illum_correct_df_method_clm_name = (
    f"illumination{column_name_separator}correction{column_name_separator}method"
)

# illumination correction metadata dataframe - offset for illumination correction column name - this is the name
# of the column to be added to the metadata dataframe to indicate the offset used for illumination correction.
illum_correct_df_offset_clm_name = (
    f"illumination{column_name_separator}correction{column_name_separator}offset"
)

# illumination correction metadata dataframe - rescale for illumination correction column name - this is the name
# of the column to be added to the metadata dataframe to indicate if the background function was rescaled when
# doing illumination correction and the method used
illum_correct_df_rescale_clm_name = f"illumination{column_name_separator}correction{column_name_separator}rescale{column_name_separator}background"

# illumination correction metadata dataframe - clipping for illumination correction column name - this is the name
# of the column to be added to the metadata dataframe to indicate if doing illumination correction the corrected image
# has been clipped.
illum_correct_df_clipping_clm_name = f"illumination{column_name_separator}correction{column_name_separator}clip{column_name_separator}output"

# illumination correction metadata dataframe - clipping min value for illumination correction column name - this is
# the name of the column to be added to the metadata dataframe to indicate what was the minimum value used
# for clipping the corrected image (if clipping was done).
illum_correct_df_clip_min_value_clm_name = f"illumination{column_name_separator}correction{column_name_separator}min{column_name_separator}clip{column_name_separator}value"

# illumination correction metadata dataframe - clipping max value for illumination correction column name - this is
# the name of the column to be added to the metadata dataframe to indicate what was the maximum value used
# for clipping the corrected image (if clipping was done).
illum_correct_df_clip_max_value_clm_name = f"illumination{column_name_separator}correction{column_name_separator}max{column_name_separator}clip{column_name_separator}value"

# illumination correction metadata dataframe - offset_background for illumination correction column name - this is
# the name of the column to be added to the metadata dataframe to indicate if doing illumination correction the
# background function has been offsetted.
illum_correct_df_offset_background_clm_name = f"illumination{column_name_separator}correction{column_name_separator}offset{column_name_separator}background"


# =============================================================================
# METADATA DATAFRAME FILE SAVING
# =============================================================================

# illumination correction metadata dataframe - computation date format - this is the format to be used for
# indicating the date when the illumination correction was computed. This is used for saving the date in the
# metadata dataframe
illum_correct_df_meta_date_format = "%y%m%d"

# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from either part4a or part3 notebook
metadata_directory = "data/proc/proc_metadata"

# processing metadata dataframe name - date format
metadata_date_format = "%Y%m%d"

# project
project_name = "ACID"

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# include indexes when saving pandas dataframes as csv files
save_csv_index = (
    False  # if False, the index will not be saved as a separate column in the csv file
)

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}4b.csv"

In [ ]:
config

In [ ]:
import pandas as pd


mydict = {"a": [1, 2, 3], "b": [4, 5, 6], fov_column_name: ["A", None, "C"]}

mydict_df = pd.DataFrame(mydict)
mydict_df.head()

In [ ]:
def apply_background_correction(image, background, **kwargs):

    background_corrected_image = correct_background(
        image=image,
        background=background,
        method=kwargs[method],
        channel_axis=kwargs[channel_axis],
        offset=kwargs[offset],
        epsilon=kwargs[epsilon],
        working_dtype=kwargs[working_dtype],
        output_dtype=kwargs[output_dtype],
        rescale_background=kwargs[rescale_background],
        clip_corrected_image=kwargs[clip_corrected_image],
        min_clip_value=kwargs[min_clip_value],
        max_clip_value=kwargs[max_clip_value],
        zero_kwargs=kwargs[zero_kwargs],
        offset_background=kwargs[offset_background],
        verbose=kwargs[verbose],
    )

    return background_corrected_image

In [ ]:
a = "illumination_correction_date"
a = a.replace("_", "HHH")
a

In [ ]:
def build_metadata_dataframe_column_naming(config):
    sep = config.column_name_separator

    return {
        "illum_correct_df_date_clm_name": config.illum_correct_df_date_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_file_name_clm_name": config.illum_correct_df_file_name_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_method_clm_name": config.illum_correct_df_method_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_offset_clm_name": config.illum_correct_df_offset_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_rescale_clm_name": config.illum_correct_df_rescale_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_clipping_clm_name": config.illum_correct_df_clipping_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_clip_min_value_clm_name": config.illum_correct_df_clip_min_value_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_clip_max_value_clm_name": config.illum_correct_df_clip_max_value_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_offset_background_clm_name": config.illum_correct_df_offset_background_clm_name.replace(
            "_", sep
        ),
    }

In [ ]:
background_correction_columns = build_metadata_dataframe_column_naming(
    config.background_correction
)
background_correction_columns

In [ ]:
# FUTURE: src/acid/io/image_loading.py

from pathlib import Path
import tifffile


def load_tiff(file_path, **kwargs):
    """Load a TIFF image from disk."""
    return tifffile.imread(file_path, **kwargs)


# FUTURE: src/acid/image_processing/background/apply_background_correction.py
def load_field_of_view(filename, config, **kwargs):
    """Load a field-of-view TIFF image from disk."""

    file_path = Path(config.fov_directory) / filename
    logging.debug(f"Load field of view from: {file_path} ")

    try:
        return load_tiff(file_path, **kwargs)
    except Exception as error:
        raise OSError(f"Could not load field of view TIFF: {file_path}") from error

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py

from datetime import datetime
from pathlib import Path

from acid.image_processing.extract_metadata import extract_ometif_imagej_metadata
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict


def build_image_metadata(field_of_view_file, config):
    field_of_view_path = Path(config.fov_directory) / str(field_of_view_file)

    image_metadata = extract_ometif_imagej_metadata(field_of_view_path)

    processing_metadata = {
        config.proc_img_meta_date_name: datetime.now().strftime(
            config.processing_date_format
        ),
        config.proc_img_meta_dtype_name: config.output_dtype,
    }

    background_correction_metadata = {
        config.illum_corr_method_metadata_entry: config.method,
        config.illum_corr_offset_metadata_entry: config.offset,
        config.illum_corr_rescale_metadata_entry: config.rescale_background,
        config.illum_corr_clipping_metadata_entry: config.clip_corrected_image,
        config.illum_corr_clip_min_value_metadata_entry: config.min_clip_value,
        config.illum_corr_clip_max_value_metadata_entry: config.max_clip_value,
        config.illum_corr_offset_background_metadata_entry: config.offset_background,
    }

    image_metadata.update(imagej_compatible_metadata_dict(processing_metadata))
    image_metadata.update(
        imagej_compatible_metadata_dict(background_correction_metadata)
    )

    return image_metadata

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py

from acid.image_processing.background.load_background_function import (
    BackgroundFunctionStrategy,
)


def get_background_for_fov(metadata_row, backgrounds, config):
    strategy = config.background_function_strategy

    if strategy == BackgroundFunctionStrategy.DATASET:
        return backgrounds

    if strategy == BackgroundFunctionStrategy.WELL:
        well = metadata_row[config.well_column_name]
        return backgrounds[well]

    if strategy == BackgroundFunctionStrategy.GRID_POSITION:
        grid_position = metadata_row[config.gridpos_column_name]
        return backgrounds[grid_position]

    raise ValueError(
        f"Invalid background_function_strategy: {strategy}. "
        "Please select either 1, 2 or 3."
    )

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py

from acid.image_processing.correct_background import correct_background


def correct_background_image(image, background, config):
    """Apply background correction to one field-of-view image."""
    correction_kwargs = {
        "method": config.method,
        "channel_axis": config.channel_axis,
        "offset": config.offset,
        "epsilon": config.epsilon,
        "working_dtype": config.working_dtype,
        "output_dtype": config.output_dtype,
        "rescale_background": config.rescale_background,
        "clip_corrected_image": config.clip_corrected_image,
        "min_clip_value": config.min_clip_value,
        "max_clip_value": config.max_clip_value,
        "zero_kwargs": config.zero_kwargs,
        "offset_background": config.offset_background,
        "verbose": config.verbose,
    }

    return correct_background(
        image=image,
        background=background,
        **correction_kwargs,
    )

In [ ]:
# FUTURE: move to src/acid/image_processing/background/apply_background_correction.py


from pathlib import Path


def make_output_filename(field_of_view_file, config):
    """Create the output filename for a background-corrected field of view."""
    field_of_view_file = Path(field_of_view_file)

    suffix = config.ome_suffix
    stem = field_of_view_file.name.removesuffix(suffix)
    print(f"Stem: {stem}")

    return (
        f"{stem}"
        f"{config.save_file_name_separator}"
        f"{config.fov_illumin_corrected_savingword}"
        f"{suffix}"
    )

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py

from pathlib import Path

from acid.utils.save_image import tifffile_save_ometiff


def save_corrected_image(output_filename, corrected_image, image_metadata, config):
    """Save one background-corrected field-of-view image."""

    output_path = Path(config.output_directory) / str(output_filename)

    output_path.parent.mkdir(parents=True, exist_ok=True)

    tifffile_save_ometiff(
        output_path,
        data=corrected_image,
        imagej=config.save_imagej_compatible,
        photometric=config.photometric,
        metadata=image_metadata,
    )

    return output_path

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py

from datetime import datetime


def make_success_result(row_index, field_of_view_file, output_file, config):

    current_date = datetime.now().strftime(config.illum_correct_df_meta_date_format)

    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "output_file": output_file,
        "success": True,
        "stage": None,
        "error_type": None,
        "error_message": None,
        config.illum_correct_df_date_clm_name: current_date,
        config.illum_correct_df_file_name_clm_name: output_file,
        config.illum_correct_df_method_clm_name: config.method,
        config.illum_correct_df_offset_clm_name: config.offset,
        config.illum_correct_df_rescale_clm_name: config.rescale_background,
        config.illum_correct_df_clipping_clm_name: config.clip_corrected_image,
        config.illum_correct_df_clip_min_value_clm_name: config.min_clip_value,
        config.illum_correct_df_clip_max_value_clm_name: config.max_clip_value,
        config.illum_correct_df_offset_background_clm_name: config.offset_background,
    }

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py


def make_failure_result(row_index, field_of_view_file, error, config, stage=None):
    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "output_file": None,
        "success": False,
        "stage": stage,
        "error_type": type(error).__name__,
        "error_message": str(error),
        config.illum_correct_df_date_clm_name: config.null_value,
        config.illum_correct_df_file_name_clm_name: config.null_value,
        config.illum_correct_df_method_clm_name: config.null_value,
        config.illum_correct_df_offset_clm_name: config.null_value,
        config.illum_correct_df_rescale_clm_name: config.null_value,
        config.illum_correct_df_clipping_clm_name: config.null_value,
        config.illum_correct_df_clip_min_value_clm_name: config.null_value,
        config.illum_correct_df_clip_max_value_clm_name: config.null_value,
        config.illum_correct_df_offset_background_clm_name: config.null_value,
    }

In [ ]:
import pandas as pd


def get_field_of_view_file(metadata_row, config):
    """Return the FOV filename from a metadata row.

    Raises:
        ValueError: If the FOV filename cell is empty.
    """
    field_of_view_file = metadata_row.get(config.fov_column_name)

    if pd.isna(field_of_view_file) or str(field_of_view_file).strip() == "":
        raise ValueError(f"Missing FOV filename in column {config.fov_column_name!r}")

    return str(field_of_view_file).strip()

In [ ]:
# apply_background_correction_batch
#   -> coordinates many FOVs

# apply_background_correction_for_fov
#   -> coordinates one FOV: load, metadata, background, correction, save

# correct_background_image
#   -> only performs image correction math


# FUTURE: src/acid/image_processing/background/apply_background_correction.py


def apply_background_correction_for_fov(row_index, metadata_row, backgrounds, config):
    # 1. Extract the filename from the row
    try:
        field_of_view_file = get_field_of_view_file(metadata_row, config)
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=None,
            error=error,
            config=config,
            stage="get_field_of_view_file",
        )

    # 2. Load the field of view
    try:
        field_of_view = load_field_of_view(field_of_view_file, config)
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=field_of_view_file,
            error=error,
            config=config,
            stage="load_field_of_view",
        )

    # 3. Update the image metadata
    image_metadata = build_image_metadata(field_of_view_file, config)

    # 4. Choose the background function depending on the strategy
    try:
        background = get_background_for_fov(metadata_row, backgrounds, config)
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=field_of_view_file,
            error=error,
            config=config,
            stage="get_background_for_fov",
        )

    # 5. Applied background correction
    try:
        corrected_image = correct_background_image(
            image=field_of_view,
            background=background,
            config=config,
        )
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=field_of_view_file,
            error=error,
            config=config,
            stage="correct_background_image",
        )

    # 6. Save corrected image
    output_filename = make_output_filename(field_of_view_file, config)

    try:
        save_corrected_image(output_filename, corrected_image, image_metadata, config)
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=field_of_view_file,
            error=error,
            config=config,
            stage="save_corrected_image",
        )

    return make_success_result(row_index, field_of_view_file, output_filename, config)

In [ ]:
from tqdm.notebook import tqdm


def apply_background_correction_batch(metadata_df, backgrounds, config, max_rows=None):
    row_indices = metadata_df.index

    if max_rows is not None:
        row_indices = metadata_df.index[:max_rows]

    results = []

    for row_index in tqdm(row_indices, desc="Applying background correction"):
        result = apply_background_correction_for_fov(
            row_index=row_index,
            metadata_row=metadata_df.loc[row_index],
            backgrounds=backgrounds,
            config=config,
        )

        results.append(result)

    return results


results = apply_background_correction_batch(
    metadata_df=metadata_df,
    backgrounds=backgrounds,
    config=config.background_correction,
)


# apply_results_to_metadata_df(metadata_df, results)
# save_metadata_df(metadata_df, config)

In [ ]:
results

In [ ]:
import pandas as pd


def get_correction_metadata_columns(config):
    return [
        config.illum_correct_df_date_clm_name,
        config.illum_correct_df_file_name_clm_name,
        config.illum_correct_df_method_clm_name,
        config.illum_correct_df_offset_clm_name,
        config.illum_correct_df_rescale_clm_name,
        config.illum_correct_df_clipping_clm_name,
        config.illum_correct_df_clip_min_value_clm_name,
        config.illum_correct_df_clip_max_value_clm_name,
        config.illum_correct_df_offset_background_clm_name,
    ]


def update_metadata_with_correction_results(metadata_df, results, config, copy=True):
    if copy:
        metadata_df = metadata_df.copy()

    if not results:
        return metadata_df

    metadata_columns = get_correction_metadata_columns(config)

    results_df = pd.DataFrame.from_records(results).set_index("row_index")

    missing_result_columns = [
        column for column in metadata_columns if column not in results_df.columns
    ]

    if missing_result_columns:
        raise KeyError(
            f"Correction results are missing metadata columns: {missing_result_columns}"
        )

    updates_df = results_df[metadata_columns]

    missing_columns = [
        column for column in metadata_columns if column not in metadata_df.columns
    ]

    metadata_df = metadata_df.assign(**{column: pd.NA for column in missing_columns})

    metadata_df = metadata_df.astype(dict.fromkeys(metadata_columns, "object"))

    metadata_df.loc[updates_df.index, metadata_columns] = updates_df.to_numpy()

    return metadata_df

In [ ]:
results

In [ ]:
metadata_df_updated = update_metadata_with_correction_results(
    metadata_df=metadata_df,
    results=results,
    config=config.background_correction,
)

In [ ]:
metadata_df_updated.head(10)

In [ ]:
metadata_df.head(10)

## 6. Update Processing Metadata Dataframe

In [ ]:
# # ---------
# # UPDATE PROCESSING METADATA DATA FRAME
# # ---------
# metadata_df[illum_correct_df_date_clm_name] = illum_correct_date_collection
# metadata_df[illum_correct_df_file_name_clm_name] = illum_correct_file_name_collection
# metadata_df[illum_correct_df_method_clm_name] = illum_correct_method_collection
# metadata_df[illum_correct_df_offset_clm_name] = illum_correct_offset_collection
# metadata_df[illum_correct_df_rescale_clm_name] = illum_correct_rescale_collection
# metadata_df[illum_correct_df_clipping_clm_name] = illum_correct_clipping_collection
# metadata_df[illum_correct_df_clip_min_value_clm_name] = (
#     illum_correct_clip_min_value_collection
# )
# metadata_df[illum_correct_df_clip_max_value_clm_name] = (
#     illum_correct_clip_max_value_collection
# )
# metadata_df[illum_correct_df_offset_background_clm_name] = (
#     illum_correct_offset_background_collection
# )

# # save the updated metadata dataframe as a csv file
# metadata_df_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
# metadata_df.to_csv(
#     os.path.join(metadata_directory, metadata_df_name), index=save_csv_index
# )

# # print progress update
# print("metadata saved")
# print("finished")